---

## 💻 Práctica


---

### 🔹 Ejercicio 1 — Sistema de pedidos con `case class`



In [1]:
case class Cliente(id: Int, nombre: String, email: String)

case class LineaPedido(producto: String, cantidad: Int, precioUnitario: Double) {
  def subtotal(): Double = cantidad * precioUnitario
}

case class Pedido(numero: Int, cliente: Cliente, lineas: List[LineaPedido]) {
  def total(): Double = lineas.map(_.subtotal()).sum

  def resumen(): String = {
    val cab  = s"=== Pedido #$numero — ${cliente.nombre} ==="
    val body = lineas.map(l =>
      f"  ${l.producto}%-20s ${l.cantidad}%2d ud × ${l.precioUnitario}%7.2f € = ${l.subtotal()}%8.2f €"
    ).mkString("\n")
    val pie  = f"  ${"TOTAL"}%-20s ${""}%2s     ${""}%10s   ${total()}%8.2f €"
    s"$cab\n$body\n${"-" * 55}\n$pie"
  }
}


val cliente1 = Cliente(1, "Ana García", "ana@ejemplo.com")

val pedido1 = Pedido(
  numero  = 1001,
  cliente = cliente1,
  lineas  = List(
    LineaPedido("Teclado mecánico",  1, 79.99),
    LineaPedido("Ratón inalámbrico", 2, 29.99),
    LineaPedido("Alfombrilla",       1, 12.50)
  )
)
println(pedido1.resumen())

// Aplicar descuento del 10% en el precio del teclado con copy
val tecladoRebajado  = pedido1.lineas.head.copy(precioUnitario = 71.99)
val pedido1Rebajado  = pedido1.copy(
  lineas = tecladoRebajado :: pedido1.lineas.tail
)

println(pedido1Rebajado.resumen())
println(s"Ahorro: ${f"${pedido1.total() - pedido1Rebajado.total()}%.2f"} €")

//Nuevo pedido con el mismo número y cliente, pero con la alfombrilla a 1.0 €
val pedido2 = pedido1.copy(
  numero = pedido1.numero + 1, // Cambiamos el número para que sea un nuevo pedido
  lineas = pedido1.lineas.map {
    case l if l.producto == "Alfombrilla" => l.copy(precioUnitario = l.precioUnitario * 0.9) // Si lo encuentra, pone el nuevo
    case l => l                  // Si no, deja el que estaba
  }
)

println(pedido2.resumen)
println(s"Ahorro: ${f"${pedido1.total() -pedido2.total()}%.2f"} €")


1 deprecation (since 2.13.3); re-run enabling -deprecation for details, or try -help


=== Pedido #1001 — Ana García ===
  Teclado mecánico      1 ud ×   79.99 € =    79.99 €
  Ratón inalámbrico     2 ud ×   29.99 € =    59.98 €
  Alfombrilla           1 ud ×   12.50 € =    12.50 €
-------------------------------------------------------
  TOTAL                                      152.47 €
=== Pedido #1001 — Ana García ===
  Teclado mecánico      1 ud ×   71.99 € =    71.99 €
  Ratón inalámbrico     2 ud ×   29.99 € =    59.98 €
  Alfombrilla           1 ud ×   12.50 € =    12.50 €
-------------------------------------------------------
  TOTAL                                      144.47 €
Ahorro: 8.00 €
=== Pedido #1002 — Ana García ===
  Teclado mecánico      1 ud ×   79.99 € =    79.99 €
  Ratón inalámbrico     2 ud ×   29.99 € =    59.98 €
  Alfombrilla           1 ud ×   11.25 € =    11.25 €
-------------------------------------------------------
  TOTAL                                      151.22 €
Ahorro: 1.25 €


defined class Cliente
defined class LineaPedido
defined class Pedido
cliente1: Cliente = Cliente(
  id = 1,
  nombre = "Ana García",
  email = "ana@ejemplo.com"
)
pedido1: Pedido = Pedido(
  numero = 1001,
  cliente = Cliente(id = 1, nombre = "Ana García", email = "ana@ejemplo.com"),
  lineas = List(
    LineaPedido(
      producto = "Teclado mecánico",
      cantidad = 1,
      precioUnitario = 79.99
    ),
    LineaPedido(
      producto = "Ratón inalámbrico",
      cantidad = 2,
      precioUnitario = 29.99
    ),
    LineaPedido(producto = "Alfombrilla", cantidad = 1, precioUnitario = 12.5)
  )
)
tecladoRebajado: LineaPedido = LineaPedido(
  producto = "Teclado mecánico",
  cantidad = 1,
  precioUnitario = 71.99
)
pedido1Rebajado: Pedido = Pedido(
  numero = 1001,
  cliente = Cliente(id = 1, nombre = "Ana García", email = "ana@ejemplo.com"),
  lineas = List(
    LineaPedido(
      producto = "Teclado mecánico",
      cantidad = 1,
      precioUnitario = 71.99
    ),
    LineaPedido

---

### 🔹 Ejercicio 2 — Pattern matching sobre una jerarquía `sealed trait`


In [2]:
sealed trait EstadoEnvio

case object Preparando                                          extends EstadoEnvio
case class  EnTransito(transportista: String, ubicacion: String) extends EstadoEnvio
case class  Entregado(firma: String)                            extends EstadoEnvio
case class  Incidencia(motivo: String)                          extends EstadoEnvio
case object Devuelto                                            extends EstadoEnvio

def descripcionEstado(estado: EstadoEnvio): String = estado match {
  case Preparando              => "📦 El paquete está siendo preparado en el almacén"
  case EnTransito(trans, ubic) => s"🚚 En tránsito con $trans — Última ubicación: $ubic"
  case Entregado(firma)        => s"✅ Entregado y firmado por: $firma"
  case Incidencia(motivo)      => s"⚠️  Incidencia: $motivo"
  case Devuelto                => "↩️  El paquete ha sido devuelto al remitente"
}


val historial: List[(String, EstadoEnvio)] = List(
  ("10:00", Preparando),
  ("14:30", EnTransito("MRW", "Madrid - Centro de clasificación")),
  ("18:00", EnTransito("MRW", "Barcelona - Almacén local")),
  ("09:15", Entregado("L. Martínez"))
)

println("=== Seguimiento de envío #ES123456 ===")
for ((hora, estado) <- historial) {
  println(s"  [$hora] ${descripcionEstado(estado)}")
}



=== Seguimiento de envío #ES123456 ===
  [10:00] 📦 El paquete está siendo preparado en el almacén
  [14:30] 🚚 En tránsito con MRW — Última ubicación: Madrid - Centro de clasificación
  [18:00] 🚚 En tránsito con MRW — Última ubicación: Barcelona - Almacén local
  [09:15] ✅ Entregado y firmado por: L. Martínez


defined trait EstadoEnvio
defined object Preparando
defined class EnTransito
defined class Entregado
defined class Incidencia
defined object Devuelto
defined function descripcionEstado
historial: List[(String, EstadoEnvio)] = List(
  ("10:00", Preparando),
  (
    "14:30",
    EnTransito(
      transportista = "MRW",
      ubicacion = "Madrid - Centro de clasificación"
    )
  ),
  (
    "18:00",
    EnTransito(transportista = "MRW", ubicacion = "Barcelona - Almacén local")
  ),
  ("09:15", Entregado(firma = "L. Martínez"))
)

---

### 🔹 Ejercicio 3 — Calculadora con pattern matching

In [3]:
sealed trait Operacion
case class Sumar(a: Double, b: Double)        extends Operacion
case class Restar(a: Double, b: Double)       extends Operacion
case class Multiplicar(a: Double, b: Double)  extends Operacion
case class Dividir(a: Double, b: Double)      extends Operacion
case class Potencia(base: Double, exp: Int)   extends Operacion


def calcular(op: Operacion): String = op match {
  case Sumar(a, b)          => f"$a + $b = ${a + b}%.4f"
  case Restar(a, b)         => f"$a - $b = ${a - b}%.4f"
  case Multiplicar(a, b)    => f"$a × $b = ${a * b}%.4f"
  case Dividir(_, 0)        => "❌ Error: división por cero"
  case Dividir(a, b)        => f"$a ÷ $b = ${a / b}%.4f"
  case Potencia(base, exp)  =>
    val resultado = scala.math.pow(base, exp)
    f"$base ^ $exp = $resultado%.4f"
}

val operaciones = List(
  Sumar(15.0, 7.5),
  Restar(100.0, 43.25),
  Multiplicar(4.0, 3.5),
  Dividir(22.0, 7.0),
  Dividir(5.0, 0.0),
  Potencia(2.0, 10)
)

println("=== Calculadora Scala ===")
for (op <- operaciones) println(s"  ${calcular(op)}")


=== Calculadora Scala ===
  15.0 + 7.5 = 22.5000
  100.0 - 43.25 = 56.7500
  4.0 × 3.5 = 14.0000
  22.0 ÷ 7.0 = 3.1429
  ❌ Error: división por cero
  2.0 ^ 10 = 1024.0000


defined trait Operacion
defined class Sumar
defined class Restar
defined class Multiplicar
defined class Dividir
defined class Potencia
defined function calcular
operaciones: List[Product with Operacion with Serializable] = List(
  Sumar(a = 15.0, b = 7.5),
  Restar(a = 100.0, b = 43.25),
  Multiplicar(a = 4.0, b = 3.5),
  Dividir(a = 22.0, b = 7.0),
  Dividir(a = 5.0, b = 0.0),
  Potencia(base = 2.0, exp = 10)
)

---

### 🔹 Ejercicio 4 — Sistema de notificaciones


In [ ]:
case class Usuario(id: Int, nombre: String, email: String)

sealed trait Notificacion
case class Email(destinatario: Usuario,
                 asunto: String,
                 cuerpo: String)         extends Notificacion
case class SMS(telefono: String,
               mensaje: String)          extends Notificacion
case class Push(dispositivo: String,
                titulo: String,
                texto: String)           extends Notificacion
case object SinNotificacion              extends Notificacion

def enviarNotificacion(n: Notificacion): String = n match {
  case Email(user, asunto, cuerpo) => s"📧 EMAIL → ${user.email}\n   Asunto: $asunto\n   Cuerpo: ${cuerpo.take(40)}..."
  case SMS(tel, msg) =>               s"📱 SMS → $tel\n   Mensaje: ${msg.take(50)}"
  case Push(disp, titulo, texto) =>   s"🔔 PUSH → $disp\n   $titulo: $texto"
  case SinNotificacion =>             "🔕 Sin notificación configurada"
}

def prioridad(n: Notificacion): Int = n match {
  case Email(_, _, _)  => 3
  case SMS(_, _)       => 2
  case Push(_, _, _)   => 1
  case SinNotificacion => 0
}

val usuario1 = Usuario(1, "Ana López",   "ana@ejemplo.com")
val usuario2 = Usuario(2, "Carlos Ruiz", "carlos@ejemplo.com")

val notificaciones: List[Notificacion] = List(
  Email(usuario1, "Confirmación de pedido", "Tu pedido #1001 ha sido confirmado y está siendo preparado."),
  SMS("+34 600 123 456", "Tu paquete llegará mañana entre las 9h y las 14h."),
  Push("iPhone-Carlos", "Nueva oferta", "30% de descuento en tecnología este fin de semana"),
  SinNotificacion
)

println("=== Centro de notificaciones ===\n")
for (n <- notificaciones.sortBy(-prioridad(_))) {
  println(enviarNotificacion(n))
  println()
}


=== Centro de notificaciones ===

📧 EMAIL → ana@ejemplo.com
   Asunto: Confirmación de pedido
   Cuerpo: Tu pedido #1001 ha sido confirmado y est...

📱 SMS → +34 600 123 456
   Mensaje: Tu paquete llegará mañana entre las 9h y las 14h.

🔔 PUSH → iPhone-Carlos
   Nueva oferta: 30% de descuento en tecnología este fin de semana

🔕 Sin notificación configurada



defined class Usuario
defined trait Notificacion
defined class Email
defined class SMS
defined class Push
defined object SinNotificacion
defined function enviarNotificacion
defined function prioridad
usuario1: Usuario = Usuario(
  id = 1,
  nombre = "Ana López",
  email = "ana@ejemplo.com"
)
usuario2: Usuario = Usuario(
  id = 2,
  nombre = "Carlos Ruiz",
  email = "carlos@ejemplo.com"
)
notificaciones: List[Notificacion] = List(
  Email(
    destinatario = Usuario(
      id = 1,
      nombre = "Ana López",
      email = "ana@ejemplo.com"
    ),
    asunto = "Confirmación de pedido",
    cuerpo = "Tu pedido #1001 ha sido confirmado y está siendo preparado."
  ),
  SMS(
    telefono = "+34 600 123 456",
    mensaje = "Tu paquete llegará mañana entre las 9h y las 14h."
  ),
  Push(
    dispositivo = "iPhone-Carlos",
    titulo = "Nueva oferta",
    texto = "30% de descuento en tecnología este fin de semana"
  ),
  SinNotificacion
)

---
# Ejercicios propuestos:

<aside>

Crea un ejemplo propio de cada sección tratada en este tema y explica linea por linea cada uno de tus ejemplos

</aside>